# Attention Residuals: a three-way comparison

This notebook compares three residual mechanisms on small GPT-2-style, byte-level language models:

| Variant | Residual mechanism | Source |
|---|---|---|
| **Baseline** | standard PreNorm residuals (fixed unit weights) | `GPT`, lucidrains' reference code (vendored at `external/deep-cross-attention/`) |
| **DCA** | DeepCrossAttention: GRN-gated access to past layer outputs (Heddes et al., ICML 2025) | `DCAGPT`, same vendored package |
| **AttnRes** | Full Attention Residuals: softmax attention over all past layer outputs (Kimi Team 2025) | `KARGPT`, implemented from scratch in `src/kar_gpt.py` |

The Kimi Team's core claim is that PreNorm residuals accumulate every layer's output with fixed weight 1, so hidden-state magnitudes grow with depth and deeper layers must produce ever-larger outputs to stay influential ("PreNorm dilution"). AttnRes replaces the fixed sum with learned, input-dependent softmax attention over depth. We check four of the paper's claims at toy scale:

1. **Loss** — comparable or better validation loss (§5.1 of the paper; expected to be noisy at this scale).
2. **Output magnitudes across depth** — the baseline's sublayer output magnitudes should grow with depth, AttnRes should stay bounded (Fig. 5b).
3. **Gradient magnitudes across depth** — the baseline should concentrate large gradients in early layers, AttnRes should be more uniform (Fig. 5c).
4. **Learned depth-attention patterns** — which earlier outputs each layer attends to (Fig. 8).

All three models share the same backbone (PreNorm attention + feed-forward blocks, RoPE, RMSNorm), hyperparameters, seed, and batch order, so differences are attributable to the residual mechanism. The training loop is adapted from the vendored `train.py`.

## Setup

Seeds are fixed and the device is auto-detected. The full run was designed for a GPU; on CPU/Apple Silicon expect it to be slow (see the config cell's quick mode).

In [ ]:
import gzip, math, os, sys, time
from pathlib import Path

import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

sys.path.insert(0, 'src')
sys.path.insert(0, 'external/deep-cross-attention')

from deep_cross_attention import GPT, DCAGPT
from kar_gpt import KARGPT

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"torch {torch.__version__} | device: {DEVICE}")

### Configuration

`NB_QUICK=1` in the environment switches to a tiny smoke-test configuration (small model, 30 steps) that exercises every cell in a couple of minutes — useful to verify the notebook runs end-to-end before committing to the full training run. The full configuration mirrors the vendored `train.py` defaults (dim 512, depth 6, seq len 512), with the batch count reduced to keep the three-model run tractable.

In [ ]:
QUICK = os.environ.get('NB_QUICK') == '1'

SEED = 42

# model (matches lucidrains' train.py defaults)
NUM_TOKENS = 256          # byte-level vocab
DIM        = 64  if QUICK else 512
DEPTH      = 3   if QUICK else 6
DIM_HEAD   = 16  if QUICK else 64
HEADS      = 4   if QUICK else 8

# training
SEQ_LEN            = 128 if QUICK else 512
BATCH_SIZE         = 4   if QUICK else 8
NUM_BATCHES        = 30  if QUICK else 2000
LR                 = 1e-4
GRAD_CLIP          = 0.5
VALIDATE_EVERY     = 10  if QUICK else 100
RECORD_GRADS_EVERY = 5   if QUICK else 20

# evaluation
NUM_VAL_BATCHES = 2 if QUICK else 8    # fixed set used during training
FINAL_EVAL_N    = 4 if QUICK else 64   # fixed set used for the final loss table

print('quick smoke-test mode' if QUICK else 'full configuration')

## Data: enwik8

We use enwik8 (the first 100M bytes of a 2006 English Wikipedia XML dump), the same corpus the vendored `train.py` trains on and a standard byte-level language-modeling benchmark. Tokens are raw bytes, so the vocabulary is 256 and no tokenizer is needed. Following `train.py`, we take the first 95M bytes and split them 90M train / 5M validation.

`data/enwik8.gz` (~35 MB) is committed to the repo so the notebook also runs on machines without internet access; the cell below re-downloads it only if it's missing.

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
ENWIK8_GZ = DATA_DIR / 'enwik8.gz'

if not ENWIK8_GZ.exists():
    import urllib.request
    url = 'https://github.com/lucidrains/deep-cross-attention/raw/main/data/enwik8.gz'
    print(f'downloading {url} ...')
    urllib.request.urlretrieve(url, ENWIK8_GZ)

with gzip.open(ENWIK8_GZ) as f:
    data = np.frombuffer(f.read(int(95e6)), dtype=np.uint8).copy()

np_train, np_valid = np.split(data, [int(90e6)])
data_train, data_val = torch.from_numpy(np_train), torch.from_numpy(np_valid)
print(f'train: {data_train.size(0):,} bytes | val: {data_val.size(0):,} bytes')

In [ ]:
class TextSamplerDataset(Dataset):
    # from the vendored train.py: every __getitem__ draws a random window
    # of SEQ_LEN + 1 bytes (inputs plus one shifted target)

    def __init__(self, data, seq_len):
        super().__init__()
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return self.data.size(0) // self.seq_len

    def __getitem__(self, index):
        rand_start = torch.randint(0, self.data.size(0) - self.seq_len, (1,))
        full_seq = self.data[rand_start : rand_start + self.seq_len + 1].long()
        return full_seq

def sample_val_batches(n, generator):
    # a fixed, pre-sampled validation set so every model is scored on identical data
    batches = []
    for _ in range(n):
        starts = torch.randint(0, data_val.size(0) - SEQ_LEN - 1, (BATCH_SIZE,), generator = generator)
        batches.append(torch.stack([data_val[s : s + SEQ_LEN + 1] for s in starts]).long())
    return batches

g = torch.Generator().manual_seed(SEED)
VAL_BATCHES = sample_val_batches(NUM_VAL_BATCHES, g)
FINAL_EVAL_SET = sample_val_batches(FINAL_EVAL_N, g)
print(f'{NUM_VAL_BATCHES} tracking batches + {FINAL_EVAL_N} final-eval batches, batch size {BATCH_SIZE}, seq len {SEQ_LEN}')

## Models

All three share the same backbone: PreNorm blocks of causal self-attention (RoPE, `F.scaled_dot_product_attention`) followed by a GELU feed-forward, RMSNorm everywhere, and a weight-untied output head. They differ only in how a layer's input is assembled from what came before:

- **Baseline (`GPT`)** — `h ← h + f(h)`: every sublayer output is added onto a single running stream with weight 1.
- **DCA (`DCAGPT`)** — each block builds *separate* Q/K/V inputs as GRN-weighted (input-dependent, ReLU-gated) combinations of the first/last `k=2` past layer outputs.
- **AttnRes (`KARGPT`)** — each sublayer input is `Σᵢ αᵢ·vᵢ` over *all* past sublayer outputs plus the embedding, with `αᵢ = softmax(wᵀ RMSNorm(vᵢ))` and a learned per-sublayer pseudo-query `w` initialized to zero (training starts from a uniform average). There is no additive residual stream at all.

Parameter counts differ slightly (DCA's GRNs and AttnRes' pseudo-queries + key-norms); at this scale AttnRes' overhead is ~0.5% of parameters — worth remembering that the paper calls it "negligible" at 48B scale, where it genuinely is.

In [ ]:
def build_model(name):
    torch.manual_seed(SEED)  # identical init RNG stream per variant
    common = dict(num_tokens = NUM_TOKENS, dim = DIM, depth = DEPTH, dim_head = DIM_HEAD, heads = HEADS)
    if name == 'Baseline':
        return GPT(**common)
    if name == 'DCA':
        return DCAGPT(**common, past_layers_k = 2)
    if name == 'AttnRes':
        return KARGPT(**common)
    raise ValueError(name)

MODEL_NAMES = ['Baseline', 'DCA', 'AttnRes']

for name in MODEL_NAMES:
    n_params = sum(p.numel() for p in build_model(name).parameters())
    print(f'{name:>9}: {n_params:,} params')

## Training

The loop below is the vendored `train.py` adapted for this notebook:

- **dropped**: `fire` (we call `train()` programmatically), `accelerate` (single device), gradient accumulation and periodic sampling (not needed for this comparison);
- **kept**: Adam at `1e-4`, gradient-norm clipping at `0.5`, random-window batches;
- **added**: instrumentation. Every `RECORD_GRADS_EVERY` steps we record each transformer block's parameter-gradient norm **before clipping** (the paper's Fig. 5c analog), and every `VALIDATE_EVERY` steps the loss on the fixed validation batches.

`torch.manual_seed(SEED)` is called right before each model's loop, and batch sampling draws from the global RNG (no dropout or other RNG consumers in the forward pass), so **all three models see exactly the same batches in the same order**.

In [ ]:
def cycle(loader):
    while True:
        for batch in loader:
            yield batch

def transformer_blocks(model):
    # per-block parameter grouping, in depth order; for KARGPT a block also
    # contains its two AttnRes ops, for DCAGPT its GRNs
    return model.dca_blocks if isinstance(model, DCAGPT) else model.layers

def block_grad_norms(blocks):
    return [
        math.sqrt(sum(p.grad.pow(2).sum().item() for p in blk.parameters() if p.grad is not None))
        for blk in blocks
    ]

@torch.no_grad()
def mean_loss(model, batches):
    model.eval()
    return float(np.mean([model(b.to(DEVICE), return_loss = True).item() for b in batches]))

In [ ]:
def train(model, num_batches = NUM_BATCHES):
    model.to(DEVICE)
    opt = Adam(model.parameters(), lr = LR)
    blocks = list(transformer_blocks(model))

    torch.manual_seed(SEED)  # -> identical batch order across models
    loader = cycle(DataLoader(TextSamplerDataset(data_train, SEQ_LEN), batch_size = BATCH_SIZE))

    hist = {'train_loss': [], 'val_step': [], 'val_loss': [], 'grad_step': [], 'grad_norms': []}

    for step in tqdm(range(1, num_batches + 1), desc = type(model).__name__):
        model.train()
        batch = next(loader).to(DEVICE)

        loss = model(batch, return_loss = True)
        opt.zero_grad()
        loss.backward()

        if step % RECORD_GRADS_EVERY == 0:
            hist['grad_step'].append(step)
            hist['grad_norms'].append(block_grad_norms(blocks))

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

        hist['train_loss'].append(loss.item())
        if step % VALIDATE_EVERY == 0:
            hist['val_step'].append(step)
            hist['val_loss'].append(mean_loss(model, VAL_BATCHES))

    hist['grad_norms'] = np.array(hist['grad_norms'])
    return hist

In [ ]:
models, histories = {}, {}

for name in MODEL_NAMES:
    model = build_model(name)
    t0 = time.time()
    histories[name] = train(model)
    models[name] = model
    print(f'{name:>9}: {time.time() - t0:5.0f}s | final tracked val loss {histories[name]["val_loss"][-1]:.4f}')

In [ ]:
# shared plot style: fixed color per variant (validated categorical palette),
# recessive grid and axes, direct end-of-line labels next to a legend

COLORS = {'Baseline': '#2a78d6', 'DCA': '#1baf7a', 'AttnRes': '#eda100'}
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'text.color': INK, 'font.size': 11, 'lines.linewidth': 2,
    'legend.frameon': False,
})

def end_labels(ax, items):
    # direct labels at the line ends, nudged apart vertically so converging
    # curves stay readable; items: list of (x, y, name)
    ax.autoscale_view()
    lo, hi = ax.get_ylim()
    gap = 0.06 * (hi - lo)
    prev = None
    for x, y, name in sorted(items, key = lambda t: t[1]):
        y_lbl = y if prev is None else max(y, prev + gap)
        prev = y_lbl
        ax.annotate(name, (x, y_lbl), xytext = (6, 0), textcoords = 'offset points',
                    color = COLORS[name], va = 'center', fontweight = 'bold', fontsize = 10)
    if prev is not None and prev + gap > hi:
        ax.set_ylim(lo, prev + gap)  # keep nudged labels inside the axes

## Eval 1: loss

Left: training loss, lightly drawn raw with a moving average on top. Right: loss on the fixed validation batches.

**Caveat:** at this scale the loss comparison is *noisy* — the paper's gaps are ~0.02 nats for models with hundreds of millions of parameters trained on tens of billions of tokens; our run covers a few million tokens on ~25M-parameter models with a single seed. "Comparable loss, trains stably" is the honest bar here (and the final table below reports the batch-level standard deviation to make the noise visible). The structural claims in Evals 2–4 are where toy scale can genuinely reproduce the paper.

In [ ]:
def smooth(x, frac = 0.02):
    w = max(1, int(len(x) * frac))
    return np.convolve(x, np.ones(w) / w, mode = 'valid')

from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 2, figsize = (12, 4))
train_ends, val_ends = [], []

for name in MODEL_NAMES:
    h, c = histories[name], COLORS[name]
    axes[0].plot(h['train_loss'], color = c, alpha = 0.15, linewidth = 1)
    s = smooth(h['train_loss'])
    axes[0].plot(np.arange(len(s)) + (len(h['train_loss']) - len(s)), s, color = c)
    train_ends.append((len(h['train_loss']) - 1, s[-1], name))

    axes[1].plot(h['val_step'], h['val_loss'], color = c, marker = 'o', markersize = 4, label = name)
    val_ends.append((h['val_step'][-1], h['val_loss'][-1], name))

end_labels(axes[0], train_ends)
end_labels(axes[1], val_ends)
axes[0].set(title = 'Training loss', xlabel = 'step', ylabel = 'cross-entropy (nats/byte)')
axes[1].set(title = 'Validation loss (fixed batches)', xlabel = 'step')
axes[1].xaxis.set_major_locator(MaxNLocator(integer = True))
axes[1].legend(loc = 'upper right')
fig.tight_layout()

In [ ]:
print(f'final loss over {FINAL_EVAL_N} fixed validation batches (mean ± std across batches):\n')
for name, model in models.items():
    model.eval()
    with torch.no_grad():
        losses = [model(b.to(DEVICE), return_loss = True).item() for b in FINAL_EVAL_SET]
    mean, std = np.mean(losses), np.std(losses)
    print(f'{name:>9}: {mean:.4f} ± {std:.4f} nats/byte  ({mean / math.log(2):.4f} bits/byte)')

## Eval 2: sublayer output magnitudes across depth

Paper claim (Fig. 5b): with standard residuals, hidden-state magnitude grows with depth, so deeper layers are "compelled to learn increasingly large outputs to remain influential" against their fixed-scale normalized inputs. AttnRes removes the unbounded accumulation, so output magnitudes should stay bounded.

We measure the mean per-token L2 norm of every attention and feed-forward sublayer's output (the layer's *contribution* `f(·)`, not the accumulated stream) on the fixed validation batches, via forward hooks — no model code is touched.

In [ ]:
def sublayer_modules(model):
    # each variant's attention + feed-forward modules, in depth order
    mods = []
    if isinstance(model, DCAGPT):
        for blk in model.dca_blocks:
            mods += [blk.attn, blk.ff]
    elif isinstance(model, KARGPT):
        for _attn_res, attn, _ff_res, ff in model.layers:
            mods += [attn, ff]
    else:
        for attn, ff in model.layers:
            mods += [attn, ff]
    return mods

@torch.no_grad()
def output_magnitudes(model):
    mods = sublayer_modules(model)
    sums = np.zeros(len(mods))

    def make_hook(idx):
        def hook(module, args, out):
            sums[idx] += out.detach().float().norm(dim = -1).mean().item()
        return hook

    handles = [m.register_forward_hook(make_hook(i)) for i, m in enumerate(mods)]
    model.eval()
    for b in VAL_BATCHES:
        model(b.to(DEVICE))
    for h in handles:
        h.remove()
    return sums / len(VAL_BATCHES)

fig, ax = plt.subplots(figsize = (8, 4.5))
xs = np.arange(1, 2 * DEPTH + 1)
ends = []
for name, model in models.items():
    mags = output_magnitudes(model)
    ax.plot(xs, mags, color = COLORS[name], marker = 'o', markersize = 4, label = name)
    ends.append((xs[-1], mags[-1], name))

end_labels(ax, ends)
ax.set(title = 'Sublayer output magnitude across depth',
       xlabel = 'sublayer index (attn, mlp alternating)',
       ylabel = 'mean per-token L2 norm')
ax.set_xticks(xs)
ax.legend(loc = 'upper left')
fig.tight_layout()

## Eval 3: gradient magnitudes across depth

Paper claim (Fig. 5c): with all residual weights fixed to 1, the baseline has no way to regulate gradient flow across depth, so the earliest layers receive disproportionately large gradients; AttnRes' learned softmax weights make the distribution substantially more uniform.

We recorded each block's parameter-gradient norm (pre-clipping) every `RECORD_GRADS_EVERY` steps during training; the plot averages over the second half of training, once dynamics have settled.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 4.5))
xs = np.arange(1, DEPTH + 1)
ends = []
for name in MODEL_NAMES:
    gn = histories[name]['grad_norms']
    settled = gn[len(gn) // 2 :].mean(axis = 0)
    ax.plot(xs, settled, color = COLORS[name], marker = 'o', markersize = 4, label = name)
    ends.append((xs[-1], settled[-1], name))

end_labels(ax, ends)
ax.set(title = 'Per-block parameter gradient norm (mean over 2nd half of training, pre-clipping)',
       xlabel = 'transformer block index', ylabel = 'gradient L2 norm')
ax.set_xticks(xs)
ax.legend(loc = 'upper right')
fig.tight_layout()

## Eval 4: learned depth-attention patterns

The paper's Fig. 8 visualizes the learned weights `α_{i→l}`: how each layer distributes its attention over previous sources. Their observations: **locality** (each layer attends most to its immediate predecessor), **embedding persistence** (source 0 keeps non-trivial weight throughout), and occasional learned skip connections.

`AttentionResidual` in `src/kar_gpt.py` can record its softmax weights (averaged over batch and sequence positions) during a forward pass. Rows are consuming ops (each block's attn and mlp, plus the final aggregation before the head); columns are sources (the embedding plus every earlier sublayer output); the upper triangle is structurally empty (a layer cannot attend to what hasn't been computed). Note the first row is trivially 1 — softmax over a single source.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

kar = models['AttnRes']

ops, row_labels = [], []
for i, (attn_res, _attn, ff_res, _ff) in enumerate(kar.layers):
    ops += [attn_res, ff_res]
    row_labels += [f'L{i + 1} attn', f'L{i + 1} mlp']
ops.append(kar.final_attn_res)
row_labels.append('output')

for op in ops:
    op.store_attn = True
with torch.no_grad():
    kar.eval()
    kar(VAL_BATCHES[0].to(DEVICE))
for op in ops:
    op.store_attn = False

n_sources = 2 * DEPTH + 1
weights = np.full((len(ops), n_sources), np.nan)
for r, op in enumerate(ops):
    w = op.stored_attn.float().cpu().numpy()
    weights[r, : len(w)] = w

col_labels = ['emb'] + row_labels[:-1]

BRAND_BLUES = LinearSegmentedColormap.from_list('brand_blues', [
    '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'])

fig, ax = plt.subplots(figsize = (9, 6))
im = ax.imshow(weights, cmap = BRAND_BLUES, vmin = 0, aspect = 'auto')
ax.set_xticks(range(n_sources), col_labels, rotation = 45, ha = 'right', fontsize = 9)
ax.set_yticks(range(len(ops)), row_labels, fontsize = 9)
ax.set(title = 'Learned depth-attention weights α (row: consumer, column: source)',
       xlabel = 'source', ylabel = 'consuming op')
ax.grid(False)
fig.colorbar(im, ax = ax, label = 'mean attention weight', shrink = 0.8)
fig.tight_layout()

## Discussion

What to look for in the results above, claim by claim:

1. **Loss** — all three variants should train stably to similar loss; given the scale and single seed, ordering differences within ~1 std are noise, as flagged in Eval 1. This matches the proposal's target of "comparable training loss" rather than reproducing the paper's compute-advantage numbers.
2. **Output magnitudes** — the baseline curve should trend upward with depth (PreNorm dilution in miniature), while AttnRes stays flat/bounded; DCA also controls growth via its gated access. This is a structural property of the update rule, so it reproduces even at toy scale.
3. **Gradient magnitudes** — the baseline should be top-heavy at the earliest blocks; AttnRes flatter. Also structural.
4. **Depth-attention heatmap** — expect diagonal dominance (locality), persistent weight on the embedding column, and possibly a few off-diagonal concentrations (learned skips) — the qualitative signatures of the paper's Fig. 8.

**Limitations worth stating plainly:** single seed; ~25M-parameter models for a few thousand steps vs. the paper's multi-billion-parameter, trillion-token runs; the paper's headline model uses *Block* AttnRes (out of scope here — we implement Full AttnRes); and AttnRes' parameter overhead (~0.5% here) is only truly negligible at scale.

*References:* Kimi Team 2025, "Attention Residuals" (Technical Report) · Heddes et al., ICML 2025, "DeepCrossAttention" · lucidrains' `deep-cross-attention` (vendored).